In [96]:
import pandas as pd
import ast

PATH = "../data/raw/listings_raw.csv"
REPORTS = "../reports/"
OUTPUT = "../data/raw/listings.csv"


In [97]:
df = pd.read_csv(PATH)

In [98]:
df.isna().sum()

url              0
fetch_date       0
title            0
location         0
house_type      18
bathrooms        1
bedrooms         1
properties       0
amenities     2245
price            0
dtype: int64

In [99]:
def get_loc(loc: pd.DataFrame):
    parts = str(loc).split(",")
    if len(parts) == 3:
        return parts[1].strip()
    elif len(parts) >= 4:
        return parts[1].strip() + ", " + parts[2].strip()
    elif len(parts) < 3:
        return parts[0]


In [100]:
df["location"] = df["location"].apply(get_loc)

In [101]:
def fix_ht(df):
    mht = df["house_type"].isna().sum()
    print(f"Missing house_type data before update: {mht}")
    df.loc[df["house_type"].isna(), "house_type"] = "Bedsitter"
    mht = df["house_type"].isna().sum()
    print(f"Missing house_type data after update nans: {mht}")
    return df

In [102]:
df = fix_ht(df)

Missing house_type data before update: 18
Missing house_type data after update nans: 0


In [103]:
def fix_bednbath(df):
    print(f"Missing bathroom data before update: {df['bathrooms'].isna().sum()}")
    print(f"Missing bedroom data before update: {df['bedrooms'].isna().sum()}")

    df.dropna(subset=["bathrooms"], inplace=True)
    df.dropna(subset=["bedrooms"], inplace=True)

    df["bathrooms"] = df["bathrooms"].str.split(" ").str.get(0).str.strip()
    df["bedrooms"] = df["bedrooms"].str.split(" ").str.get(0).str.strip()

    print(f"Missing bathroom data after update: {df['bathrooms'].isna().sum()}")
    print(f"Missing bedroom data after update: {df['bedrooms'].isna().sum()}")

    return df

In [104]:
df = fix_bednbath(df)

Missing bathroom data before update: 1
Missing bedroom data before update: 1
Missing bathroom data after update: 0
Missing bedroom data after update: 0


In [105]:
def extract_all_attributes(df, drop=True):
    """Expand the property column into separate columns"""
    olen = len(df.columns)

    df["properties"] = df["properties"].apply(ast.literal_eval)
    properties = df["properties"].apply(pd.Series)
    df_prop = df.join(properties)
    if drop:
        df_prop = df_prop.drop("properties", axis=1)

    plen = len(df_prop.columns)

    print(f"Properties extracted; added {plen - olen} new attributes!")

    amenities = df_prop["amenities"].str.strip().str.get_dummies(sep=",")
    df_amen = df_prop.join(amenities)
    if drop:
        df_amen = df_amen.drop("amenities", axis=1)

    alen = len(df_amen.columns)
    print(f"Ameneties extracted; added {alen - plen} new attributes!")
    print(f"{alen} final attributes after extraction")

    facilities = df_amen["Facilities"].str.strip().str.get_dummies(sep=",")
    df_amen.update(facilities)  # Updates overlapping columns in-place
    df_fac = df_amen

    return df_fac

In [106]:
df = extract_all_attributes(df)

Properties extracted; added 23 new attributes!
Ameneties extracted; added 17 new attributes!
50 final attributes after extraction


In [107]:
df["price"] = df["price"].str.replace("GH₵ ", "").str.replace(",", "").astype(float)

In [108]:
def clean_self_contained(df):
    """
    Clean and fix the 'Self Contained' column based on sequential rules.
    """
    df = df.copy()

    # Rule 1: Old condition AND Shared Apartment -> No
    df.loc[
        (df["Condition"] == "Old") & (df["house_type"] == "Shared Apartment"),
        "Self Contained",
    ] = "No"

    # Rule 2: Shared Apartment -> No
    df.loc[
        (df["house_type"] == "Shared Apartment"),
        "Self Contained",
    ] = "No"

    # Rule 3: Price >= 1000 -> Yes
    df.loc[
        (df["price"] >= 1000),
        "Self Contained",
    ] = "Yes"

    # Rule 4: Old condition -> No
    df.loc[
        (df["Condition"] == "Old"),
        "Self Contained",
    ] = "No"

    # Rule 5: Wi-Fi available -> Yes
    df.loc[
        (df["Wi-Fi"] == 1),
        "Self Contained",
    ] = "Yes"

    # Rule 6: Has amenities/furnishing -> Yes
    df.loc[
        (df["Apartment"] == 1)
        | (df["Wardrobe"] == 1)
        | (df["TV"] == 1)
        | (df["Furnishing"] == "Furnished")
        | (df["Facilities"].notna())
        | (df["Dishwasher"] == 1)
        | (df["Dining Area"] == 1)
        | (df["Kitchen Cabinets"] == 1)
        | (df["Kitchen Shelf"] == 1),
        "Self Contained",
    ] = "Yes"

    # Rule 7: Low price AND no agency fee -> No
    df.loc[
        (df["price"] <= 400) & (df["Agency Fee"].isna()),
        "Self Contained",
    ] = "No"

    # Rule 8: Has caution fee -> Yes
    df.loc[
        (df["Caution Fee"].notna()),
        "Self Contained",
    ] = "Yes"

    # Rule 9: Low price AND no agency fee -> No (duplicate of Rule 7)
    df.loc[
        (df["price"] <= 400) & (df["Agency Fee"].isna()),
        "Self Contained",
    ] = "No"

    # Rule 10: Medium-low price AND has agency fee -> Yes
    df.loc[
        (df["price"] <= 500) & (df["Agency Fee"].notna()),
        "Self Contained",
    ] = "Yes"

    # Rule 11: Price >= 500 -> Yes
    df.loc[
        (df["price"] >= 500),
        "Self Contained",
    ] = "Yes"

    # Rule 12: Price < 500 -> No
    df.loc[
        (df["price"] < 500),
        "Self Contained",
    ] = "No"

    return df

In [109]:
df = clean_self_contained(df)

In [110]:
df.to_csv(OUTPUT)